# Muffled Voice Distress Classifier — Training Pipeline

**Picks up from ChatGPT Step 7.**

CNVVE non-distress hard negatives should already be processed at:
`/content/muffled_training/processed/cnvve_negative/`

This notebook:
1. Loads Vocal Bursts dataset for distress source audio
2. Extracts cry/scream/whimper/moan/gasp classes
3. Generates muffled distress clips via augmentation
4. Combines with CNVVE non-distress
5. Trains a binary muffled distress classifier
6. Exports as TFLite + PyTorch

**GPU required:** Runtime → Change runtime type → T4 GPU

In [ ]:
#@title 0. Install dependencies
!pip install -q datasets librosa soundfile scikit-learn torch torchaudio

import os, json, random, shutil, warnings
from pathlib import Path
import numpy as np
warnings.filterwarnings('ignore')

BASE = Path('/content/muffled_training')
RAW = BASE / 'raw'
PROCESSED = BASE / 'processed'
GENERATED = BASE / 'generated'
SPLITS = BASE / 'splits'
MODELS = BASE / 'models'
REPORTS = BASE / 'reports'

for d in [RAW / 'distress_source', RAW / 'non_distress_source',
          PROCESSED / 'cnvve_negative', PROCESSED / 'muffled_distress',
          PROCESSED / 'muffled_non_distress', PROCESSED / 'final',
          GENERATED / 'muffled_distress', GENERATED / 'muffled_non_distress',
          SPLITS, MODELS, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

print('Base:', BASE)
print('GPU:', os.environ.get('COLAB_GPU', 'none'))

In [ ]:
#@title 1. Verify CNVVE non-distress is ready
cnvve_dir = PROCESSED / 'cnvve_negative'
cnvve_files = list(cnvve_dir.glob('*.wav'))
print(f'CNVVE non-distress clips: {len(cnvve_files)}')

if len(cnvve_files) == 0:
    print('\n⚠️  CNVVE not found. Run the ChatGPT steps first.')
    print('   Re-run the CNVVE processing cells before continuing.')
else:
    print('✅ CNVVE ready.')

## Step 7 — Load Vocal Bursts Dataset

In [ ]:
#@title 7. Load Vocal Bursts from HuggingFace
from datasets import load_dataset

vocal_bursts = load_dataset('TTS-AGI/vocal-bursts')
print(vocal_bursts)

for split in vocal_bursts:
    print(f'\nSPLIT: {split}')
    print(f'  Columns: {vocal_bursts[split].column_names}')
    print(f'  Rows: {len(vocal_bursts[split])}')
    print(f'  Example: {vocal_bursts[split][0]}')

## Step 8 — Inspect Vocal Bursts Labels

In [ ]:
#@title 8. Find all label columns in Vocal Bursts
for split in vocal_bursts:
    ds = vocal_bursts[split]
    print(f'\n{"=" * 40}\n  {split}\n{"=" * 40}')
    for col in ds.column_names:
        print(f'\n--- {col} ---')
        try:
            vals = ds[col]
            if isinstance(vals[0], str):
                from collections import Counter
                counts = Counter(vals)
                for v, c in counts.most_common(50):
                    print(f'  {v:30s} {c:5d}')
            else:
                print(f'  type={type(vals[0]).__name__}, example={vals[0]}')
        except Exception as e:
            print(f'  Error: {e}')

In [ ]:
#@title 8b. Auto-detect the label column (run after inspecting above)
# Adjust this based on what you see in Step 8 output
# Common column names: 'emotion', 'label', 'category', 'vocalization'

LABEL_COL = None  # ← SET THIS after inspecting Step 8 output

# Auto-detect: look for a column with string values
if LABEL_COL is None:
    for split in vocal_bursts:
        for col in vocal_bursts[split].column_names:
            vals = vocal_bursts[split][col]
            if isinstance(vals[0], str) and len(set(vals)) < 100:
                LABEL_COL = col
                break
        if LABEL_COL:
            break

if LABEL_COL:
    print(f'Detected label column: "{LABEL_COL}"')
    all_labels = set()
    for split in vocal_bursts:
        all_labels.update(vocal_bursts[split][LABEL_COL])
    print(f'All labels ({len(all_labels)}):')
    for l in sorted(all_labels):
        print(f'  {l}')
else:
    print('Could not auto-detect. Set LABEL_COL manually above.')

In [ ]:
#@title 8c. Define which labels are distress
# Adjust these lists based on the actual labels you saw in Step 8b

DISTRESS_LABELS = {
    'cry', 'crying',
    'scream', 'screaming',
    'whimper', 'whimpering',
    'moan', 'moaning',
    'gasp', 'gasping',
    'groan', 'groaning',
    'sob', 'sobbing',
    'wail', 'wailing',
    'pain',  # some datasets use this
    'fear', 'fright',
    'distress',
}

NON_DISTRESS_LABELS = {
    'laugh', 'laughter', 'happy', 'joy',
    'neutral', 'calm',
    'speech', 'talking',
    'hum', 'humming',
    'ahem',
    'cough',
    'sigh',
    'yawn',
}

# Auto-match what actually exists in the dataset
if LABEL_COL:
    matched_distress = set()
    matched_non_distress = set()
    all_labels = set()
    for split in vocal_bursts:
        all_labels.update(vocal_bursts[split][LABEL_COL])
    for l in all_labels:
        ll = l.lower().strip()
        if ll in DISTRESS_LABELS or any(d in ll for d in ['cry', 'scream', 'whimper', 'moan', 'gasp', 'groan', 'sob', 'wail', 'pain', 'fear', 'distress']):
            matched_distress.add(l)
        elif ll in NON_DISTRESS_LABELS or any(n in ll for n in ['laugh', 'happy', 'joy', 'neutral', 'calm', 'speech', 'hum', 'ahem', 'cough']):
            matched_non_distress.add(l)

    print('Matched DISTRESS labels:')
    for l in sorted(matched_distress):
        count = sum(len([x for x in vocal_bursts[s][LABEL_COL] if x == l]) for s in vocal_bursts)
        print(f'  {l:30s} {count:5d}')

    print(f'\nMatched NON-DISTRESS labels:')
    for l in sorted(matched_non_distress):
        count = sum(len([x for x in vocal_bursts[s][LABEL_COL] if x == l]) for s in vocal_bursts)
        print(f'  {l:30s} {count:5d}')

## Step 9 — Extract Distress Source Audio

In [ ]:
#@title 9. Download distress clips from Vocal Bursts to disk
import librosa
import soundfile as sf

distress_dir = RAW / 'distress_source'
distress_dir.mkdir(parents=True, exist_ok=True)

extracted = 0
skipped = 0

for split_name in vocal_bursts:
    ds = vocal_bursts[split_name]
    for i, row in enumerate(ds):
        label = row[LABEL_COL]
        if label not in matched_distress:
            continue

        # Vocal Bursts stores audio as dict with 'array' and 'sampling_rate'
        audio_data = row.get('audio', None)
        if audio_data is None:
            skipped += 1
            continue

        try:
            audio = np.array(audio_data['array'], dtype=np.float32)
            sr = audio_data['sampling_rate']

            # Resample to 16kHz mono
            if sr != 16000:
                audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
            if audio.ndim > 1:
                audio = audio.mean(axis=1)

            # Normalize
            peak = np.max(np.abs(audio))
            if peak > 0:
                audio = audio / peak * 0.95

            safe_label = label.lower().replace(' ', '_').replace('-', '_')
            out_path = distress_dir / f'{safe_label}_{split_name}_{i:06d}.wav'
            sf.write(str(out_path), audio, 16000)
            extracted += 1
        except Exception as e:
            skipped += 1

print(f'Extracted: {extracted} distress clips')
print(f'Skipped: {skipped}')

# Show breakdown by label
from collections import Counter
label_counts = Counter()
for f in distress_dir.glob('*.wav'):
    label = f.name.split('_')[0]
    label_counts[label] += 1
print('\nPer-label counts:')
for label, count in sorted(label_counts.items()):
    print(f'  {label:25s} {count:5d}')

## Step 10 — Listen to samples

In [ ]:
#@title 10. Listen to extracted distress clips
from IPython.display import Audio, display

distress_files = list(distress_dir.glob('*.wav'))
samples = random.sample(distress_files, min(8, len(distress_files)))

for f in samples:
    print(f'{f.name} ({f.stat().st_size // 1024}KB)')
    display(Audio(str(f)))

## Step 11 — Muffled Distress Augmentation

In [ ]:
#@title 11. Muffled distress augmentation pipeline
SR = 16000

def muffle_audio(audio, sr=SR, severity='medium'):
    """
    Simulate muffled audio as if mouth is obstructed.
    
    What happens physically when something is in the mouth:
    - High frequencies are attenuated (obstruction blocks them)
    - Overall amplitude drops
    - Formant structure distorts (mouth can't open fully)
    - More breathiness relative to voicing
    - Fundamental frequency is preserved but weaker
    """
    x = audio.copy().astype(np.float32)
    rng = np.random.default_rng()

    # Severity presets
    presets = {
        'light':  {'lp_cutoff': 2000, 'gain_db': -6,  'noise_mix': 0.02, 'formant_boost': 400},
        'medium': {'lp_cutoff': 1200, 'gain_db': -10, 'noise_mix': 0.05, 'formant_boost': 350},
        'heavy':  {'lp_cutoff': 800,  'gain_db': -15, 'noise_mix': 0.08, 'formant_boost': 300},
        'extreme': {'lp_cutoff': 500,  'gain_db': -20, 'noise_mix': 0.12, 'formant_boost': 250},
    }
    p = presets.get(severity, presets['medium'])

    # 1. Low-pass filter (the obstruction acts as a physical LPF)
    cutoff = p['lp_cutoff'] + rng.uniform(-200, 200)
    # Simple first-order IIR low-pass
    alpha = cutoff / (cutoff + sr / (2 * np.pi))
    filtered = np.empty_like(x)
    state = 0.0
    for i in range(len(x)):
        state += alpha * (x[i] - state)
        filtered[i] = state
    x = filtered

    # 2. Gain reduction
    gain = 10 ** ((p['gain_db'] + rng.uniform(-3, 3)) / 20)
    x *= gain

    # 3. Formant resonance (constrained oral cavity)
    formant_freq = p['formant_boost'] + rng.uniform(-50, 50)
    t = np.arange(len(x)) / sr
    formant = np.sin(2 * np.pi * formant_freq * t) * 0.1
    x += formant * np.abs(x)  # modulated by signal amplitude

    # 4. Breathy noise (struggling to breathe through obstruction)
    if p['noise_mix'] > 0:
        breath = rng.standard_normal(len(x)).astype(np.float32)
        # Breathy noise is mostly in 1-4 kHz
        breath_alpha = 1500 / (1500 + sr / (2 * np.pi))
        breath_filtered = np.empty_like(breath)
        bstate = 0.0
        for i in range(len(breath)):
            bstate += breath_alpha * (breath[i] - bstate)
            breath_filtered[i] = bstate
        x += breath_filtered * p['noise_mix'] * rng.uniform(0.5, 1.5)

    # 5. Random reverb (muffled sounds have less direct signal)
    if rng.random() < 0.4:
        delay = int(rng.uniform(0.005, 0.03) * sr)
        if delay < len(x):
            x[delay:] += x[:-delay] * rng.uniform(0.08, 0.25)

    # Normalize
    peak = np.max(np.abs(x))
    if peak > 0:
        x = x / peak * 0.9

    return np.clip(x, -0.98, 0.98).astype(np.float32)


print('Muffled augmentation pipeline defined.')
print('Severity levels: light, medium, heavy, extreme')

In [ ]:
#@title 11b. Generate muffled distress clips from source audio
muffled_distress_dir = GENERATED / 'muffled_distress'
muffled_distress_dir.mkdir(parents=True, exist_ok=True)

# How many muffled variants per source clip
VARIANTS_PER_CLIP = 4  # one of each severity

generated = 0
source_files = list(distress_dir.glob('*.wav'))

for src in source_files:
    try:
        audio, src_sr = librosa.load(str(src), sr=SR, mono=True)
    except Exception as e:
        print(f'ERROR loading {src.name}: {e}')
        continue

    stem = src.stem
    for variant, severity in enumerate(['light', 'medium', 'heavy', 'extreme']):
        muffled = muffle_audio(audio, SR, severity)
        out_path = muffled_distress_dir / f'{stem}_muffled_{severity}.wav'
        sf.write(str(out_path), muffled, SR)
        generated += 1

print(f'Generated {generated} muffled distress clips from {len(source_files)} sources')
print(f'Output: {muffled_distress_dir}')

# Also copy original (unmuffled) distress clips as a variant
for src in source_files:
    dst = muffled_distress_dir / f'{src.stem}_original.wav'
    shutil.copy2(src, dst)
    generated += 1

print(f'Total distress clips (including originals): {generated}')

In [ ]:
#@title 11c. Listen to muffled distress samples
from IPython.display import Audio, display

# Pick one source and show all severity levels
source_stems = list(set(f.name.split('_muffled_')[0] for f in muffled_distress_dir.glob('*_muffled_*.wav')))
if source_stems:
    pick = random.choice(source_stems)
    print(f'Comparing: {pick}\n')
    for severity in ['original', 'light', 'medium', 'heavy', 'extreme']:
        matches = list(muffled_distress_dir.glob(f'{pick}_*{severity}.wav'))
        if matches:
            print(f'  {severity}:')
            display(Audio(str(matches[0])))

## Step 12 — Build Combined Dataset

In [ ]:
#@title 12. Combine all sources into final dataset
final_dir = PROCESSED / 'final'
final_dir.mkdir(parents=True, exist_ok=True)

manifest_rows = []

# Label 1 = MUFFLED_DISTRESS (from Vocal Bursts + muffled augmentation)
distress_clips = list(muffled_distress_dir.glob('*.wav'))
for i, src in enumerate(distress_clips):
    dst = final_dir / f'distress_{i:06d}.wav'
    shutil.copy2(src, dst)
    manifest_rows.append({
        'path': str(dst),
        'label': 'muffled_distress',
        'class': 1,
        'source': src.stem.split('_')[0],
        'augmentation': '_'.join(src.stem.split('_')[2:]) if '_muffled_' in src.stem else 'original',
    })

# Label 0 = NON_DISTRESS (from CNVVE)
cnvve_clips = list(cnvve_dir.glob('*.wav'))
for i, src in enumerate(cnvve_clips):
    dst = final_dir / f'non_distress_{i:06d}.wav'
    shutil.copy2(src, dst)
    manifest_rows.append({
        'path': str(dst),
        'label': 'non_distress',
        'class': 0,
        'source': 'cnvve',
        'augmentation': 'none',
    })

# Shuffle
random.seed(42)
random.shuffle(manifest_rows)

print(f'Final dataset: {len(manifest_rows)} clips')
print(f'  Muffled distress (class 1): {sum(1 for r in manifest_rows if r["class"] == 1)}')
print(f'  Non-distress (class 0):      {sum(1 for r in manifest_rows if r["class"] == 0)}')

In [ ]:
#@title 12b. Speaker-safe train/val/test split
# Ensure no source speaker appears in multiple splits

# Group by source
by_source = {}
for row in manifest_rows:
    src = row['source']
    if src not in by_source:
        by_source[src] = []
    by_source[src].append(row)

sources = list(by_source.keys())
random.seed(42)
random.shuffle(sources)

# 70/15/15 split at the source level
n = len(sources)
n_train = int(0.7 * n)
n_val = int(0.15 * n)

train_sources = sources[:n_train]
val_sources = sources[n_train:n_train + n_val]
test_sources = sources[n_train + n_val:]

train_rows = [r for s in train_sources for r in by_source[s]]
val_rows = [r for s in val_sources for r in by_source[s]]
test_rows = [r for s in test_sources for r in by_source[s]]

print(f'Train: {len(train_rows)} clips ({len(train_sources)} sources)')
print(f'Val:   {len(val_rows)} clips ({len(val_sources)} sources)')
print(f'Test:  {len(test_rows)} clips ({len(test_sources)} sources)')

# Check no source leakage
train_src_set = set(train_sources)
val_src_set = set(val_sources)
test_src_set = set(test_sources)
assert not (train_src_set & val_src_set), 'LEAKAGE: train/val share sources'
assert not (train_src_set & test_src_set), 'LEAKAGE: train/test share sources'
assert not (val_src_set & test_src_set), 'LEAKAGE: val/test share sources'
print('\n✅ No source leakage between splits.')

# Save splits as CSV
import csv
for name, rows in [('train', train_rows), ('val', val_rows), ('test', test_rows)]:
    csv_path = SPLITS / f'{name}.csv'
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['path', 'label', 'class', 'source', 'augmentation'])
        writer.writeheader()
        writer.writerows(rows)
    print(f'Saved {csv_path} ({len(rows)} rows)')

In [ ]:
#@title 12c. Class balance check
for split_name in ['train', 'val', 'test']:
    csv_path = SPLITS / f'{split_name}.csv'
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))
    n_distress = sum(1 for r in rows if r['class'] == '1')
    n_non = sum(1 for r in rows if r['class'] == '0')
    ratio = n_distress / max(1, n_non)
    print(f'{split_name:6s}: distress={n_distress:4d}  non_distress={n_non:4d}  ratio={ratio:.2f}')

## Step 13 — Train Binary Muffled Distress Classifier

Architecture: Same compact CNN as the existing 5-class model, but output is binary (2 classes).
Input: 96×64×1 log-mel spectrogram → binary sigmoid output.

In [ ]:
#@title 13. Feature extraction + training (PyTorch)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


# --- Log-mel feature extraction (matches hub/voice_decision.py exactly) ---
def hz_to_mel(hz):
    return 2595.0 * np.log10(1.0 + hz / 700.0)

def mel_to_hz(mel):
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)

def make_mel_filterbank(n_fft=512, bands=64, sr=SR):
    hz = mel_to_hz(np.linspace(hz_to_mel(20.0), hz_to_mel(sr / 2.0), bands + 2))
    bins = np.clip(np.floor((n_fft + 1) * hz / sr).astype(int), 0, n_fft // 2)
    bank = np.zeros((bands, n_fft // 2 + 1), dtype=np.float32)
    for i in range(bands):
        left, mid, right = bins[i:i + 3]
        if mid > left:
            bank[i, left:mid] = np.linspace(0.0, 1.0, mid - left, endpoint=False)
        if right > mid:
            bank[i, mid:right] = np.linspace(1.0, 0.0, right - mid, endpoint=False)
    return bank

MEL_BANK = make_mel_filterbank()

def log_mel_feat(audio, sr=SR):
    """Deterministic 96x64 log-mel (matches hub/voice_decision.py)."""
    x = np.asarray(audio, dtype=np.float32).reshape(-1)
    # Resample if needed
    if sr != SR and x.size >= 2:
        n = max(1, round(x.size * SR / sr))
        x = np.interp(np.linspace(0, x.size - 1, n), np.arange(x.size), x).astype(np.float32)
    # Fit to 2 seconds
    target = SR * 2
    if x.size < target:
        x = np.pad(x, (0, target - x.size))
    elif x.size > target:
        x = x[-target:]
    # STFT
    frame, hop, n_fft = 400, 160, 512
    count = 1 + (x.size - frame) // hop
    frames = np.stack([x[i * hop:i * hop + frame] for i in range(count)])
    power = np.abs(np.fft.rfft(frames * np.hanning(frame), n=n_fft, axis=1)) ** 2
    mel = np.log(np.maximum(power @ MEL_BANK.T, 1e-8)).astype(np.float32)
    # Interpolate to 96 time frames
    idx = np.linspace(0, mel.shape[0] - 1, 96)
    lo = np.floor(idx).astype(int)
    hi = np.minimum(lo + 1, mel.shape[0] - 1)
    frac = (idx - lo)[:, None]
    return (mel[lo] * (1.0 - frac) + mel[hi] * frac).astype(np.float32)

print('Feature extraction defined.')

In [ ]:
#@title 13b. Dataset class + data loading
class MuffledDataset(Dataset):
    def __init__(self, csv_path, augment=False, seed=42):
        with open(csv_path) as f:
            self.rows = list(csv.DictReader(f))
        self.augment = augment
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        audio, sr = librosa.load(row['path'], sr=SR, mono=True)
        label = int(row['class'])

        if self.augment:
            # Gain augmentation
            audio *= float(self.rng.uniform(0.6, 1.3))
            # Random shift
            if self.rng.random() < 0.3:
                shift = int(self.rng.integers(0, min(len(audio), SR)))
                audio = np.roll(audio, shift)

        feat = log_mel_feat(audio, SR)
        return torch.from_numpy(feat[..., None]), torch.tensor(label, dtype=torch.long)


train_ds = MuffledDataset(SPLITS / 'train.csv', augment=True)
val_ds = MuffledDataset(SPLITS / 'val.csv', augment=False)
test_ds = MuffledDataset(SPLITS / 'test.csv', augment=False)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
#@title 13c. Model definition (compact CNN)
class MuffledDistressNet(nn.Module):
    """
    Compact CNN for binary muffled distress detection.
    Architecture matches the 5-class voice_distress model but with 2-class output.
    Input: [B, 1, 96, 64] log-mel spectrogram
    Output: [B, 2] logits
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm([1, 96, 64]),
            nn.Conv2d(1, 24, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(48, 72, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.25),
            nn.Linear(72, 2),
        )

    def forward(self, x):
        return self.net(x)


model = MuffledDistressNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')
print(model)

In [ ]:
#@title 13d. Training loop
from sklearn.metrics import classification_report, confusion_matrix

# Class weights for imbalance
n_distress = sum(1 for r in train_ds.rows if r['class'] == '1')
n_non = sum(1 for r in train_ds.rows if r['class'] == '0')
class_weights = torch.tensor([n_distress / n_non, 1.0], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

EPOCHS = 40
best_val_acc = 0.0
patience_counter = 0
MAX_PATIENCE = 10

print(f'Training for {EPOCHS} epochs (early stopping patience={MAX_PATIENCE})')
print(f'Class weights: non_distress={class_weights[0]:.3f}, distress={class_weights[1]:.3f}')
print()

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
        train_correct += (logits.argmax(1) == y).sum().item()
        train_total += x.size(0)
    train_loss /= train_total
    train_acc = train_correct / train_total

    # Validate
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item() * x.size(0)
            val_correct += (logits.argmax(1) == y).sum().item()
            val_total += x.size(0)
    val_loss /= val_total
    val_acc = val_correct / val_total

    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]['lr']

    marker = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), MODELS / 'best_muffled_distress.pth')
        marker = ' ★'
    else:
        patience_counter += 1

    print(f'Epoch {epoch+1:3d}/{EPOCHS} | train_loss={train_loss:.4f} acc={train_acc:.3f} | val_loss={val_loss:.4f} acc={val_acc:.3f} | lr={lr:.6f}{marker}')

    if patience_counter >= MAX_PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}')
        break

print(f'\nBest val accuracy: {best_val_acc:.3f}')

## Step 14 — Evaluation

In [ ]:
#@title 14. Evaluate on held-out test set
model.load_state_dict(torch.load(MODELS / 'best_muffled_distress.pth'))
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x)
        probs = F.softmax(logits, dim=1)
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())
        all_probs.extend(probs.cpu().numpy())

print('Test Set Results')
print('=' * 60)
print(classification_report(all_labels, all_preds, target_names=['non_distress', 'muffled_distress']))
print('Confusion Matrix:')
print(confusion_matrix(all_labels, all_preds))

# Find optimal threshold
distress_probs = np.array([p[1] for p in all_probs])
true_labels = np.array(all_labels)

print('\nThreshold analysis:')
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7]:
    preds_t = (distress_probs >= thresh).astype(int)
    tp = ((preds_t == 1) & (true_labels == 1)).sum()
    fp = ((preds_t == 1) & (true_labels == 0)).sum()
    fn = ((preds_t == 0) & (true_labels == 1)).sum()
    tn = ((preds_t == 0) & (true_labels == 0)).sum()
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-8, precision + recall)
    print(f'  thresh={thresh:.1f}  P={precision:.3f}  R={recall:.3f}  F1={f1:.3f}')

## Step 15 — Export

Exports:
1. **PyTorch `.pth`** — for further training/experimentation
2. **ONNX** — for cross-platform deployment
3. **TFLite** — for on-device (ESP32) or lightweight inference

In [ ]:
#@title 15a. Export as ONNX
model.load_state_dict(torch.load(MODELS / 'best_muffled_distress.pth'))
model.eval()

dummy = torch.randn(1, 1, 96, 64).to(device)
onnx_path = MODELS / 'muffled_distress.onnx'
torch.onnx.export(model, dummy, str(onnx_path),
                   input_names=['log_mel'],
                   output_names=['probs'],
                   dynamic_axes={'log_mel': {0: 'batch'}, 'probs': {0: 'batch'}})
print(f'ONNX exported: {onnx_path}')
print(f'Size: {onnx_path.stat().st_size / 1024:.1f} KB')

In [ ]:
#@title 15b. Export as TFLite (via ONNX→TFLite conversion)
# Note: Direct PyTorch→TFLite is not supported.
# Options:
#   A) Use onnx2tf: pip install onnx2tf && onnx2tf -i model.onnx
#   B) Use TensorFlow's converter directly
#   C) Re-train in TensorFlow using the same architecture

print('TFLite export options:')
print()
print('Option A (recommended):')
print('  !pip install onnx2tf')
print('  !onnx2tf -i /content/muffled_training/models/muffled_distress.onnx -o /content/muffled_training/models/')
print()
print('Option B: Re-train in TensorFlow using ml/train_voice_distress.py')
print('  Convert the train/val CSVs to the manifest format expected by voice_dataset.py')
print('  Then run: python ml/train_voice_distress.py --manifest splits/manifest.csv --output models/muffled_distress.tflite')
print()
print('Option C: Use ai_edge_torch for direct PyTorch→TFLite')
print('  !pip install ai-edge-torch')
print('  import ai_edge_torch')
print('  edge_model = ai_edge_torch.convert(model, (dummy,))')
print('  edge_model.export("muffled_distress.tflite")')

In [ ]:
#@title 15c. Save metadata + download
metadata = {
    'model': 'muffled_distress',
    'version': 'v1',
    'classes': ['non_distress', 'muffled_distress'],
    'input': 'log_mel [1, 96, 64, 1]',
    'sample_rate': SR,
    'train_samples': len(train_ds),
    'val_samples': len(val_ds),
    'test_samples': len(test_ds),
    'best_val_acc': best_val_acc,
    'threshold': 0.5,
    'pytorch_model': 'best_muffled_distress.pth',
    'onnx_model': 'muffled_distress.onnx',
}
(MODELS / 'muffled_distress_meta.json').write_text(json.dumps(metadata, indent=2))
print('Metadata saved.')

# Zip everything for download
!cd /content/muffled_training/models && zip -j /content/muffled_distress_models.zip *.pth *.onnx *.json

from google.colab import files
files.download('/content/muffled_distress_models.zip')

## Step 16 — Quick sanity test

Run the model against a few clips to verify it works.

In [ ]:
#@title 16. Sanity test: run model on random clips
from IPython.display import Audio, display

model.load_state_dict(torch.load(MODELS / 'best_muffled_distress.pth'))
model.eval()

class_names = ['NON_DISTRESS', 'MUFFLED_DISTRESS']

# Test on random clips from each split
for split_name in ['val', 'test']:
    csv_path = SPLITS / f'{split_name}.csv'
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))
    samples = random.sample(rows, min(5, len(rows)))

    print(f'\n--- {split_name.upper()} SET ---')
    for row in samples:
        audio, _ = librosa.load(row['path'], sr=SR, mono=True)
        feat = log_mel_feat(audio, SR)
        x = torch.from_numpy(feat[..., None]).unsqueeze(0).to(device)
        with torch.no_grad():
            probs = F.softmax(model(x), dim=1).cpu().numpy()[0]
        pred = probs.argmax()
        true = int(row['class'])
        correct = '✅' if pred == true else '❌'
        print(f'  {correct} true={class_names[true]:20s} pred={class_names[pred]:20s} probs=[{probs[0]:.3f}, {probs[1]:.3f}]  {Path(row["path"]).name}')

## Integration with VanniKawachh

To use this model in the hub pipeline:

1. Copy `best_muffled_distress.pth` (or ONNX/TFLite equivalent) to `hub/models/muffled_distress.pth`
2. In `hub/voice_decision.py`, add a `MuffledDistressBackend` that:
   - Loads the model
   - Runs `log_mel()` on incoming audio
   - Returns probability of `muffled_distress` class
3. In `VoiceDecisionEngine.analyse()`, add the muffled distress probability as an additional evidence source alongside the existing 5-class model and adaptive baseline

Detection paths after integration:
```
Model available:
  - Strong CNN (≥0.90)              → confirmed
  - Two moderate CNN (2× ≥0.65)     → confirmed
  - Moderate CNN + baseline          → confirmed
  - Muffled distress (≥0.70)         → confirmed (NEW)
  - Keyword + prosody/model          → confirmed

Model NOT available:
  - Baseline stress ≥ 0.70          → confirmed
  - Keyword + prosody                → confirmed
```